# 01 — Explore lateral_detection labels

Sanity-check the COCO keypoint annotations from `keypoints.v56-lat_only_baseline.coco`.

For each image we want to see, side by side:

1. **Raw image** — the blueprint at native resolution.
2. **Chord endpoints** — every 2-point annotation drawn as red dots.
3. **Reconstructed polylines** — endpoints fused within `merge_radius`, degree-2 chains walked into polylines (one color per polyline).
4. **Rasterized GT mask** — the binary `(0, 1)` mask we will train against, painted at `thickness` pixels and overlaid in red.

Use this notebook to:

- Confirm the COCO loader reads the dataset correctly.
- Pick a good value for `merge_radius` (default 10 px) and stroke `thickness` (default 4 px) by visual inspection.
- Spot systematic labelling problems (missing chords, gaps, misaligned endpoints) before they ruin training.

> Whatever values look right here go into `configs/base.yaml`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# When run from notebooks/, hop up to the repo root so that
# `from data.coco_loader import ...` resolves.
if Path.cwd().name == "notebooks":
    os.chdir("..")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print("cwd:", Path.cwd())

import yaml
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Some plans are very large; lift PIL's safety limit.
Image.MAX_IMAGE_PIXELS = None

from data.coco_loader import load_split, summarize_split, LATERAL_CATEGORY_ID
from data.polyline_builder import build_polylines
from data.rasterize import rasterize_polylines

## Load config + all three splits

Reads `configs/base.yaml`, resolves the dataset root, and loads each split. The summary block prints how many images and chord annotations each split contains, plus image-size statistics — a quick way to confirm the loader works.

In [ ]:
config = yaml.safe_load(open("configs/base.yaml"))

dataset_root = (Path.cwd() / config["data"]["dataset_root"]).resolve()
assert dataset_root.is_dir(), f"Dataset root not found: {dataset_root}"
print("dataset_root:", dataset_root)

split_dirs = {
    "train": dataset_root / config["data"]["train_dir"],
    "valid": dataset_root / config["data"]["valid_dir"],
    "test":  dataset_root / config["data"]["test_dir"],
}

split_data = {name: load_split(p) for name, p in split_dirs.items()}

for name, (imgs, chords) in split_data.items():
    s = summarize_split(imgs, chords)
    print(f"[{name:>5}] n_images={s['n_images']:>3}  n_chords={s['n_chords']:>5}  "
          f"chords/image={s['chords_per_image']}  size={s['image_size']}")

## Pick one image and load it

Change `SPLIT` / `IMG_INDEX` to inspect a different image. `MERGE_RADIUS` and `THICKNESS` start at the values from `configs/base.yaml`; tweak them here to see how the visualization reacts before committing changes to the config file.

In [ ]:
SPLIT       = "train"
IMG_INDEX   = 0
MERGE_RADIUS = config["polyline"]["merge_radius"]
THICKNESS    = config["rasterize"]["thickness"]

images, chords_by_image = split_data[SPLIT]
img_ids = sorted(images.keys())
img_id = img_ids[IMG_INDEX]
record = images[img_id]
chords = chords_by_image[img_id]

print(f"file_name:     {record.file_name}")
print(f"resolution:    {record.width} x {record.height}")
print(f"# of chords:   {len(chords)}")
print(f"path:          {record.path}")

img_rgb = np.array(Image.open(record.path).convert("RGB"))
print(f"loaded image:  shape={img_rgb.shape}, dtype={img_rgb.dtype}")

## Build polylines + rasterize the GT mask

`build_polylines` fuses endpoints within `MERGE_RADIUS`, walks degree-2 chains, and returns a list of polylines. `rasterize_polylines` paints them onto a binary mask at the configured stroke `THICKNESS`. The numbers below let us sanity-check that endpoint merging actually reconstructs continuous lines (fewer polylines than chords means chaining worked).

In [ ]:
polylines = build_polylines(chords, merge_radius=MERGE_RADIUS)
pts_per_pl = [len(pl.points) for pl in polylines]

print(f"chords:                {len(chords)}")
print(f"polylines:             {len(polylines)}")
if pts_per_pl:
    print(f"points per polyline:   "
          f"min={min(pts_per_pl)}, median={int(np.median(pts_per_pl))}, max={max(pts_per_pl)}")

mask = rasterize_polylines(polylines, record.height, record.width, thickness=THICKNESS)
fg = int((mask > 0).sum())
total = record.height * record.width
print(f"mask shape:            {mask.shape}")
print(f"foreground pixels:     {fg:,}  ({100.0 * fg / total:.3f}% of {total:,})")

## Visualization helpers

Two reasons we downsample for display only:

1. A 10800 × 7200 plan won't fit in any reasonable matplotlib figure.
2. Coordinates of chords / polylines / masks are always kept at native
   resolution; we only scale the geometry **when drawing** so the panels
   stay aligned.

The mask is rasterized at native resolution (no downsampling on the GT side); we resize the mask with `INTER_NEAREST` only for the overlay.

In [ ]:
def downscale(arr, max_side=1600):
    """Resize so the longest side is at most `max_side`. Returns (arr, scale)."""
    h, w = arr.shape[:2]
    s = max_side / max(h, w)
    if s >= 1.0:
        return arr.copy(), 1.0
    new_h, new_w = int(round(h * s)), int(round(w * s))
    interp = cv2.INTER_AREA if arr.ndim == 3 else cv2.INTER_NEAREST
    return cv2.resize(arr, (new_w, new_h), interpolation=interp), s


def draw_chord_endpoints(img_rgb, chords, scale=1.0, radius=3, color=(255, 0, 0)):
    out = img_rgb.copy()
    for c in chords:
        for pt in (c.p1 * scale, c.p2 * scale):
            cv2.circle(out, (int(pt[0]), int(pt[1])), radius, color, -1, cv2.LINE_AA)
    return out


def draw_polylines(img_rgb, polylines, scale=1.0, thickness=2, seed=0):
    out = img_rgb.copy()
    rng = np.random.default_rng(seed)
    for pl in polylines:
        pts = (pl.points * scale).astype(np.int32).reshape(-1, 1, 2)
        if pts.shape[0] < 2:
            continue
        color = tuple(int(x) for x in rng.integers(60, 255, size=3))
        cv2.polylines(out, [pts], False, color, thickness=thickness, lineType=cv2.LINE_AA)
    return out


def overlay_mask(img_rgb, mask, color=(255, 0, 0), alpha=0.55):
    """Tint pixels where `mask > 0` with `color` at the given alpha."""
    out = img_rgb.copy()
    sel = mask > 0
    for c in range(3):
        out[..., c] = np.where(
            sel,
            (1 - alpha) * out[..., c] + alpha * color[c],
            out[..., c],
        ).astype(np.uint8)
    return out


def render_panels(img_rgb, chords, polylines, mask, max_side=1600):
    img_small, scale = downscale(img_rgb, max_side=max_side)
    mask_small = cv2.resize(
        mask,
        (img_small.shape[1], img_small.shape[0]),
        interpolation=cv2.INTER_NEAREST,
    )
    return [
        img_small,
        draw_chord_endpoints(img_small, chords, scale=scale),
        draw_polylines(img_small, polylines, scale=scale),
        overlay_mask(img_small, mask_small),
    ], scale

## Four-panel inspection of the selected image

In [ ]:
panels, scale = render_panels(img_rgb, chords, polylines, mask, max_side=1600)
print(f"display downscale factor: {scale:.4f}")

titles = [
    "1. raw image",
    "2. chord endpoints (raw annotations)",
    f"3. polylines after merge_radius={MERGE_RADIUS}px  ({len(polylines)} polylines)",
    f"4. rasterized GT mask, thickness={THICKNESS}px (red overlay)",
]

fig, axes = plt.subplots(1, 4, figsize=(28, 10))
for ax, title, panel in zip(axes, titles, panels):
    ax.imshow(panel)
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()
fig.suptitle(
    f"[{SPLIT}] {record.file_name}   "
    f"({record.width} x {record.height},   "
    f"{len(chords)} chords)",
    fontsize=12,
)
fig.tight_layout()
plt.show()

## Sweep `merge_radius` to pick a good value

Re-runs polyline construction at several `merge_radius` values and overlays the result on the same image. The right value is one where consecutive chords chain together (so we don't see isolated stubs) but separate real lateral lines stay separate.

In [ ]:
sweep_radii = [0, 5, 10, 20, 40]

img_small, scale = downscale(img_rgb, max_side=1600)

fig, axes = plt.subplots(1, len(sweep_radii), figsize=(6 * len(sweep_radii), 8))
for ax, r in zip(axes, sweep_radii):
    polys_r = build_polylines(chords, merge_radius=r)
    canvas = draw_polylines(img_small, polys_r, scale=scale)
    ax.imshow(canvas)
    ax.set_title(f"merge_radius = {r}  →  {len(polys_r)} polylines")
    ax.set_axis_off()
fig.suptitle(f"merge_radius sweep — {record.file_name}", fontsize=12)
fig.tight_layout()
plt.show()

## Batch: save 4-panel overlays for many images

Renders the 4-panel comparison for the first `N_PER_SPLIT` images of each split and saves them as JPEGs under `results/data_debug/`. Useful for browsing the full dataset offline (file manager / image viewer) without re-running the notebook.

In [ ]:
OUT_DIR = Path("results/data_debug")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PER_SPLIT = 5
MAX_DISPLAY_SIDE = 1600


def render_and_save(split_name, img_id):
    images, chords_by_image = split_data[split_name]
    rec = images[img_id]
    chords_here = chords_by_image[img_id]
    img = np.array(Image.open(rec.path).convert("RGB"))
    polys = build_polylines(chords_here, merge_radius=MERGE_RADIUS)
    msk = rasterize_polylines(polys, rec.height, rec.width, thickness=THICKNESS)

    panels, _ = render_panels(img, chords_here, polys, msk, max_side=MAX_DISPLAY_SIDE)
    combined = np.concatenate(panels, axis=1)

    stem = Path(rec.file_name).stem.replace(".", "_")[:48]
    out_path = OUT_DIR / f"{split_name}_id{img_id:03d}_{stem}.jpg"
    cv2.imwrite(
        str(out_path),
        cv2.cvtColor(combined, cv2.COLOR_RGB2BGR),
        [cv2.IMWRITE_JPEG_QUALITY, 92],
    )
    return out_path, len(chords_here), len(polys), int((msk > 0).sum())


for split_name in ("train", "valid", "test"):
    images_s, _ = split_data[split_name]
    ids = sorted(images_s.keys())[:N_PER_SPLIT]
    for img_id in ids:
        out, n_c, n_p, fg = render_and_save(split_name, img_id)
        print(f"[{split_name}] id={img_id:>3}  chords={n_c:>4}  "
              f"polylines={n_p:>3}  fg_px={fg:>8}  →  {out.name}")